In [1]:
pip install torch transformers

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datetime import datetime

class TransformerChatbot:
    def __init__(self, model_name="microsoft/DialoGPT-medium"):
        print(f"Initializing {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.chat_history_ids = None
        self.step_count = 0
        print("Model loaded successfully!\n")

    def get_response(self, user_input):
        # 1. Encode user input and add EOS token
        new_user_input_ids = self.tokenizer.encode(
            user_input + self.tokenizer.eos_token,
            return_tensors='pt'
        )

        # 2. Append to history for continuous conversation
        bot_input_ids = torch.cat([self.chat_history_ids, new_user_input_ids], dim=-1) \
            if self.chat_history_ids is not None else new_user_input_ids

        # 3. Generate response with tuned parameters
        self.chat_history_ids = self.model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=self.tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            temperature=0.7
        )

        # 4. Decode the result (ignoring the previous history tokens)
        response = self.tokenizer.decode(
            self.chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        # 5. Context Management: Reset context if it gets too long to prevent memory lag
        self.step_count += 1
        if self.step_count > 7:
            self.reset_context()

        return response

    def reset_context(self):
        self.chat_history_ids = None
        self.step_count = 0

def start_app():
    # Instantiate the chatbot
    bot = TransformerChatbot()

    print("="*50)
    print("AI ASSISTANT: ONLINE")
    print("Commands: 'exit' to quit | 'reset' to clear memory")
    print("="*50)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    while True:
        # User Input Handling
        user_query = input("User: ").strip()

        # Exit Condition
        if user_query.lower() in ['exit', 'quit']:
            print(f"Chatbot: Goodbye! (Session ended at {datetime.now().strftime('%H:%M')})")
            break

        if user_query.lower() == 'reset':
            bot.reset_context()
            print("Chatbot: Memory cleared! What's next?")
            continue

        if not user_query:
            continue

        # Response Generation
        try:
            bot_response = bot.get_response(user_query)

            # Simple check for empty responses
            if not bot_response.strip():
                bot_response = "That's interesting! Tell me more."

            print(f"Chatbot: {bot_response}")

        except Exception as e:
            print(f"Chatbot Error: {e}")
            bot.reset_context()

if __name__ == "__main__":
    start_app()

Initializing microsoft/DialoGPT-medium...


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!

AI ASSISTANT: ONLINE
Commands: 'exit' to quit | 'reset' to clear memory
Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Hello , How are you
Chatbot: I'm good how are you?
User: what is  AI?
Chatbot: AI is a computer
User: what is fast api?
Chatbot: I don't know
User: whats your name
Chatbot: My name is not fast
User: exit
Chatbot: Goodbye! (Session ended at 16:27)
